# ccv on a Kaggle T4

Runs Context Cost Verifier suites against vLLM on one T4 GPU. The runner is resumable, so you can run this notebook repeatedly until every suite is complete.

**Notebook settings (right-hand panel)**
- Accelerator: **GPU T4 x2** (the notebook uses GPU 0)
- Internet: **On** (requires a phone-verified Kaggle account)

**How to run**
1. Set `MODEL` and `SUITES` in the next cell.
2. Click **Save Version → Save & Run All (Commit)**. The run continues in the background and its output is saved as a new version.
3. **To resume**, open *Add Input → Your Work → this notebook* and attach the previous version's output. The restore cell picks up `results/<model>.jsonl` from it, and runs that are already done are skipped.

Run `qwen2.5-7b` on one account and `llama3.1-8b` on a teammate's account; each writes its own JSONL file.

In [ ]:
import os, time
NOTEBOOK_START = time.time()

MODEL = "qwen2.5-7b"              # "llama3.1-8b" on the second account; "qwen2.5-0.5b" for a smoke test
SUITES = "noise shapley ablations" # use "calibrate" while building the benchmark
LIMIT = ""                         # e.g. "--limit 5" for a smoke test
SESSION_MINUTES = 540              # the GPU session limit shown by Kaggle; the runner stops safely before it
VLLM_VERSION = "0.29.0"            # pin the newest version that passes the smoke test
REPO = "https://github.com/MmohithSai/context-cost-verifier.git"
COMPRESS = False                   # True only after benchmark/tasks changed (then commit benchmark/compressed.json)
HELDOUT = False                    # True with SUITES = "heldout": run on the unused drafts in benchmark/drafts (D40)
# For a private repo, store a GitHub token as a Kaggle Secret named GITHUB_TOKEN and uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
WORK = "/kaggle/working"
CCV = "/kaggle/tmp/ccv"  # outside WORK, so the saved output is just results and logs
OUT = f"{WORK}/results/{MODEL}{'.heldout' if HELDOUT else ''}.jsonl"

## 1. Install vLLM and ccv

In [ ]:
!pip install -q "vllm=={VLLM_VERSION}"
# Kaggle preinstalls a torchaudio built for a different CUDA than vLLM's torch, and transformers
# imports torchaudio whenever it is installed, which crashes vllm serve at startup
!pip uninstall -y -q torchaudio
!git clone -q {REPO} {CCV} 2>/dev/null || git -C {CCV} pull -q
!pip install -q -e "{CCV}[gpu]"
!ccv --version && nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import shutil
assert shutil.which("ccv"), "install failed: is Internet on? (needs a phone-verified Kaggle account)"

## 2. (Optional) Refresh the LLMLingua-2 cache
Runs in its own environment, before vLLM takes the GPU. Commit the updated `benchmark/compressed.json` afterwards.

In [ ]:
if COMPRESS:
    !python -m venv --system-site-packages --without-pip /kaggle/tmp/lingua
    !/kaggle/tmp/lingua/bin/python -m pip install -q llmlingua "transformers<5"
    !/kaggle/tmp/lingua/bin/python -m pip install -q --no-deps -e {CCV}
    !cd {CCV} && /kaggle/tmp/lingua/bin/python -m ccv.cli compress --device cuda
    !cp {CCV}/benchmark/compressed.json {WORK}/compressed.json

## 3. Restore results from a previous version

In [ ]:
import glob, shutil
os.makedirs(os.path.dirname(OUT), exist_ok=True)
previous = sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(OUT)}", recursive=True), key=os.path.getsize)
if previous and (not os.path.exists(OUT) or os.path.getsize(previous[-1]) > os.path.getsize(OUT)):
    shutil.copy(previous[-1], OUT)
print(OUT, os.path.getsize(OUT) if os.path.exists(OUT) else 0, "bytes")

## 4. Start the vLLM server

In [ ]:
import subprocess, urllib.request
log = open(f"{WORK}/vllm.log", "w")
server = subprocess.Popen(["ccv", "serve", "--model", MODEL, "--", "--attention-backend", "TRITON_ATTN"],
                          stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
started = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError(open(f"{WORK}/vllm.log").read()[-4000:])
        if time.time() - started > 1500:
            raise TimeoutError("vLLM did not become healthy within 25 minutes")
        time.sleep(10)
print(f"vLLM ready after {time.time() - started:.0f}s")

## 5. Run the suites (resumable)

In [ ]:
budget = int(SESSION_MINUTES - (time.time() - NOTEBOOK_START) / 60 - 20)
print("time budget for this session:", budget, "minutes")
BENCH = f"{CCV}/benchmark"
if HELDOUT:  # the unused drafts as a benchmark of their own: tasks/ plus the shared segments
    BENCH = "/kaggle/tmp/heldout"
    !mkdir -p {BENCH} && cp -r {CCV}/benchmark/drafts {BENCH}/tasks && cp -r {CCV}/benchmark/shared {BENCH}/shared
!cd {CCV} && ccv run --model {MODEL} --suite {SUITES} --bench {BENCH} --out {OUT} --max-minutes {budget} {LIMIT}

## 6. Sanity checks

In [ ]:
import sys
from collections import Counter
sys.path.insert(0, CCV)  # the editable install's .pth is only read when Python starts, after this kernel did
from ccv.runner import read_records
records = list(read_records(OUT))
ok = [r for r in records if r["status"] == "ok"]
print(Counter((r["suite"], r["status"]) for r in records))
print("exact token attribution:", sum(bool(r["attribution_ok"]) for r in ok), "/", len(ok))
print("server timing metrics:  ", sum(bool(r["server_metrics"]) for r in ok), "/", len(ok))
print("energy measured:        ", sum(r["energy_j"] is not None for r in ok), "/", len(ok))
print("hit max_tokens:         ", sum(r["decode_tokens"] >= r["env"]["sampling"]["max_tokens"] for r in ok))  # vLLM may say "tool_calls" (D38)
for r in [r for r in records if r["status"] != "ok"][:5]:
    print("error:", r["task_id"], r["config_id"], r["error"])

In [ ]:
server.terminate()
server.wait(timeout=120)
!tail -n 5 {WORK}/vllm.log